# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdityaAAND/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Signal 1 — Volume (links to FlyRank's quick-win flag): more competing content items for a query should mean each gets a smaller share of clicks. Signal 2 — Position spread: competing pages should cluster at similar (weaker) average positions rather than one clearly winning. Rule: severity_score = content_count * (1 - top_content_click_share). Higher = more content fighting over the same query with no clear winner.

In [14]:
from huggingface_hub import HfApi
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
api = HfApi(token=hf_token)

# Check who you're logged in as
print(api.whoami())

# List files in the dataset repo
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
for f in files:
    print(f)

{'type': 'user', 'id': '6a6b62935c944c826703fb14', 'name': 'adityaanand73', 'fullname': 'Aditya Anand', 'email': 'adityaanandgzb@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/37e34179d13ddd4ea3fadedccc0e49ce.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'FLYRANKAI-Aditya _ANAND', 'role': 'read', 'createdAt': '2026-07-30T14:44:45.606Z'}}}
.gitattributes
README.md
dim_clients.parquet
dim_content.parquet
fact_content_daily_performance/month=2025-01/data_0.parquet
fact_content_daily_performance/month=2025-02/data_0.parquet
fact_content_daily_performance/month=2025-03/data_0.parquet
fact_content_daily_performance/month=2025-04/data_0.parquet
fact_content_daily_performance/month=2025-05/data_0.parquet
fact_content_daily_performance/month=2025-06/data_0.parquet
fact_content_daily_performance/month=2025-07/data_0.parquet
fact_content_daily_performance/month=2025-0

In [15]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')
print("Token loaded:", hf_token is not None, "| length:", len(hf_token) if hf_token else 0)

Token loaded: True | length: 37


In [16]:
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
print("Secret created")


Secret created


In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import duckdb

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
agg = con.sql(f"""
    SELECT client_hash_id, query_hash_id,
    COUNT(DISTINCT content_hash_id) AS content_count,
    MAX(clicks_90d) AS top_clicks,
    SUM(clicks_90d) AS total_clicks,
    AVG(avg_position_90d) AS avg_pos
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    GROUP BY client_hash_id, query_hash_id
""").df()

#Signal1:

agg['top_share'] = agg['top_clicks'] / agg['total_clicks'].replace(0,1)
bucket1 = agg.groupby(pd.cut(agg['content_count'],[0,1,2,3,5,100])).agg(
    avg_top_share = ('top_share','mean'), n=('top_share','size')

)
print(bucket1)

#Signal2:
bucket2 = agg.groupby(pd.cut(agg['content_count'], [0,1,2,3,5,100])).agg(
    avg_position=('avg_pos','mean'), n=('avg_pos','size'))
print(bucket2)





FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

               avg_top_share       n
content_count                       
(0, 1]              0.081112  829462
(1, 2]              0.149402  213314
(2, 3]              0.196008   88511
(3, 5]              0.247309   70805
(5, 100]            0.331186   57279
               avg_position       n
content_count                      
(0, 1]            16.950313  829462
(1, 2]            18.241201  213314
(2, 3]            19.830290   88511
(3, 5]            21.310243   70805
(5, 100]          23.461835   57279


/tmp/ipykernel_733/321021289.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = agg.groupby(pd.cut(agg['content_count'],[0,1,2,3,5,100])).agg(
/tmp/ipykernel_733/321021289.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket2 = agg.groupby(pd.cut(agg['content_count'], [0,1,2,3,5,100])).agg(


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
agg['severity_score'] = agg['content_count'] * (1 - agg['top_share'])

def reason_code(row):
    if row['content_count'] <= 1: return 'LOW_COUNT'
    return 'HIGH_COUNT_LOW_DOMINANCE' if row['top_share'] < 0.6 else 'HIGH_COUNT_HIGH_DOMINANCE'

def action(row):
    if row['severity_score'] > 2: return 'CONSOLIDATE'
    elif row['severity_score'] > 0.5: return 'MONITOR'
    return 'NO_ACTION'

agg['reason_code'] = agg.apply(reason_code, axis=1)
agg['action'] = agg.apply(action, axis=1)

queue = agg.sort_values('severity_score', ascending=False)
import os
os.makedirs('work/outputs', exist_ok=True)
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
queue.head(10)

,client_hash_id,query_hash_id,content_count,top_clicks,total_clicks,avg_pos,top_share,severity_score,reason_code,action
801439,client_73cda7b4e4f265ea,query_d741bc9c5f71aee4,842,89,466.0,2.514503,0.190987,681.188841,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
244453,client_8ddc46da5414ffd8,query_993729fa64134ca9,693,140,827.0,1.930915,0.169287,575.684401,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
803382,client_73cda7b4e4f265ea,query_9a77450699268f30,471,24,158.0,1.801025,0.151899,399.455696,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
868344,client_86ebc2f12c01f586,query_e2980674b49e1d64,457,15,82.0,3.236218,0.182927,373.402439,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
171664,client_73cda7b4e4f265ea,query_05118e04631b9763,398,22,208.0,2.505837,0.105769,355.903846,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
133935,client_0b245132bb722950,query_513316504a983ff7,354,0,0.0,110.931540,0.000000,354.000000,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
868265,client_86ebc2f12c01f586,query_c52e3517bb105e08,374,5,38.0,1.862125,0.131579,324.789474,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
803388,client_73cda7b4e4f265ea,query_f05bec897f35dbe1,347,4,51.0,3.150118,0.078431,319.784314,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
873931,client_8ddc46da5414ffd8,query_4c7caf726002042c,334,19,174.0,3.555031,0.109195,297.528736,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE
801540,client_73cda7b4e4f265ea,query_c26522db2bbdf9a0,362,12,65.0,8.342590,0.184615,295.169231,HIGH_COUNT_LOW_DOMINANCE,CONSOLIDATE


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Row 1 — Action: CONSOLIDATE. Why: 4 competing pages, no single page over 40% of clicks. Would be wrong if: these are actually 4 distinct search intents (e.g. "buy X" vs "review X"), not true duplication.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20)[['client_hash_id','query_hash_id','content_count','top_share',
                         'severity_score','reason_code','action']]
top20['confidence'] = top20.apply(
    lambda r: 'high' if r['content_count'] >= 3 else 'medium', axis=1)
print(top20.to_string())

                 client_hash_id           query_hash_id  content_count  top_share  severity_score               reason_code       action confidence
801439  client_73cda7b4e4f265ea  query_d741bc9c5f71aee4            842   0.190987      681.188841  HIGH_COUNT_LOW_DOMINANCE  CONSOLIDATE       high
244453  client_8ddc46da5414ffd8  query_993729fa64134ca9            693   0.169287      575.684401  HIGH_COUNT_LOW_DOMINANCE  CONSOLIDATE       high
803382  client_73cda7b4e4f265ea  query_9a77450699268f30            471   0.151899      399.455696  HIGH_COUNT_LOW_DOMINANCE  CONSOLIDATE       high
868344  client_86ebc2f12c01f586  query_e2980674b49e1d64            457   0.182927      373.402439  HIGH_COUNT_LOW_DOMINANCE  CONSOLIDATE       high
171664  client_73cda7b4e4f265ea  query_05118e04631b9763            398   0.105769      355.903846  HIGH_COUNT_LOW_DOMINANCE  CONSOLIDATE       high
133935  client_0b245132bb722950  query_513316504a983ff7            354   0.000000      354.000000  HIGH_COUNT_LO

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*
Weakest picks: rows where content_count is only 2 and top_share is close to 0.5 — borderline cases that could just be normal seasonal fluctuation rather than true cannibalization, not a strong enough signal to act on confidently. No future-window or label-derived inputs were used in the score (see leakage check above).

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no future-window or label-derived columns fed the score
used_cols = ['content_count', 'top_clicks', 'total_clicks', 'top_share']
label_or_future = ['severity_score', 'reason_code', 'action']  # these are OUTPUTS, not inputs
overlap = set(used_cols) & set(label_or_future)
print("Leakage check - overlap between inputs and outputs:", overlap if overlap else "None found")

# Confirm window_end doesn't exceed your analysis cutoff
window_check = con.sql(f"""
    SELECT MAX(window_end) AS latest_window
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
""").df()
print(window_check)

Leakage check - overlap between inputs and outputs: None found


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  latest_window
0    2026-06-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.